# Fall 2026 Capstone: Phase 1-2 (Weeks 1-4)
### Full Dataset Assembly and Infrastructure Setup

**Project:** Automated Code Documentation and Bug Report Analysis Using Fine-Tuned CodeT5+ Models
**Scope covered here:** Weeks 1-2 (dataset assembly, all 6 languages + GitBugs) and Weeks 3-4 (infrastructure, LoRA setup, locked eval/split protocol)

This notebook produces the combined dataset and pipeline that Weeks 5-8 (the CodeT5+ full fine-tune and LoRA runs) will both consume, so the train/val/test split and eval harness built here are locked and reused, not rebuilt per model.


## Week 1-2: Full Multi-Language Dataset Assembly

### Setup: install and import dependencies

In [ ]:
!pip install -q datasets transformers peft evaluate nltk sacrebleu rouge-score pandas pyarrow requests

### Mount Google Drive (run before anything else)

All outputs below are written under `/content/drive/MyDrive/capstone/data`. Without this mount, the folder is created on Colab's temporary disk and is deleted when the runtime resets.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
assert os.path.isdir('/content/drive/MyDrive'), 'Drive is not mounted, outputs would be lost on runtime reset'
print('Drive mounted.')


### Load CodeSearchNet across all six languages

Spring semester used the Python subset only. This pulls all six CodeSearchNet
languages so the Fall dataset reflects the full multi-language scope described
in the project overview.

**Loaded via `claudios/code_search_net`, a parquet reupload of the original
dataset, not `code_search_net` directly.** As of `datasets` 4.0, the library
dropped support for loading-script-based datasets entirely, so
`trust_remote_code=True` now raises `RuntimeError: Dataset scripts are no
longer supported` rather than working around it. `claudios/code_search_net`
is the same data (same fields: `func_code_string`, `func_documentation_string`,
`language`) pre-converted to parquet, so no script execution is needed.

If you hit `NameError: name 'load_dataset' is not defined`, it usually means
this notebook's setup cell (the one with the `!pip install` and imports)
wasn't run in this kernel session, most often because the runtime was
restarted after installing packages. Run the cells from the top, in order,
in a single session (Runtime -> Run all), rather than jumping straight to
this cell.

In [ ]:
from datasets import load_dataset  # re-imported here so this cell is self-contained

LANGUAGES = ["python", "java", "javascript", "go", "php", "ruby"]

raw_by_language = {}
for lang in LANGUAGES:
    print(f"Loading CodeSearchNet: {lang}")
    ds = load_dataset("claudios/code_search_net", lang, split="train")
    raw_by_language[lang] = ds
    print(f"  {lang}: {len(ds):,} raw examples")


### Unify schema: code-summarization pairs

In [ ]:
from datasets import concatenate_datasets  # re-imported here so this cell is self-contained

def unify_codesearchnet_example(example, language):
    """Map a raw CodeSearchNet record to the shared schema used across
    both the code-summarization and bug-report-summarization sources."""
    return {
        "input_text": example.get("func_code_string", "").strip(),
        "target_text": example.get("func_documentation_string", "").strip(),
        "source": "codesearchnet",
        "language": language,
        "task_type": "code_summarization",
    }

unified_code_datasets = []
for lang, ds in raw_by_language.items():
    mapped = ds.map(
        lambda ex: unify_codesearchnet_example(ex, lang),
        remove_columns=ds.column_names,
        desc=f"Unifying schema ({lang})",
    )
    unified_code_datasets.append(mapped)

code_dataset = concatenate_datasets(unified_code_datasets)
print(f"Combined CodeSearchNet (all languages): {len(code_dataset):,} examples")


### Load GitBugs data

GitBugs (Patil, 2025, arXiv:2504.09651) is CC BY 4.0 and hosted at
`github.com/av9ash/gitbugs`. Per-project CSVs are fetched directly from
GitHub (some are zipped). Each row has a `Summary` (short, one-line) and a
`Description` (long, detailed bug text) plus `Resolution`, `Status`,
`Priority`, and timestamps.

**Mozilla Core is excluded.** Its folder in the repo only ships
`Core_bugs-combined.csv`, which is Issue id / Duplicate id pairs for the
duplicate-detection task, not the underlying Summary/Description text. Pulling
the actual bug text would mean querying Bugzilla directly, which is out of
scope for this pass. That drops the usable GitBugs pool from ~150K to
roughly 107K reports across the other eight projects, which is still enough
to combine with CodeSearchNet for the ~200K target.

In [ ]:
import json
import random
import hashlib
from pathlib import Path

import pandas as pd
from datasets import load_dataset, Dataset, concatenate_datasets

random.seed(42)

DATA_DIR = Path("/content/drive/MyDrive/capstone/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import io
import zipfile
import requests

GITBUGS_RAW_BASE = "https://raw.githubusercontent.com/av9ash/gitbugs/main"

# path: repo path relative to GITBUGS_RAW_BASE
# zip_inner: filename inside the zip, if the CSV is zipped
GITBUGS_PROJECTS = {
    "cassandra":     {"path": "cassandra/cassandra_bugs.csv"},
    "firefox":       {"path": "firefox/Firefox_bugs.csv.zip", "zip_inner": "Firefox_bugs.csv"},
    "hadoop":        {"path": "hadoop/hadoop_bugs.csv"},
    "hbase":         {"path": "hbase/hbase_bugs.csv"},
    "ms_vscode_bugs":{"path": "ms_vscode_bugs/vscode_bugs.csv.zip", "zip_inner": "vscode_bugs.csv"},
    "seamonkey":     {"path": "seamonkey/SeaMonkey_bugs.csv"},
    "spark":         {"path": "spark/spark_bugs.csv"},
    "thunderbird":   {"path": "thunderbird/Thunderbird_bugs.csv"},
    # "mozillacore" intentionally omitted, see markdown above.
}

def fetch_gitbugs_project(project_key, config, cache_dir):
    """Download one project's CSV (unzipping if needed) and cache it locally
    so re-runs don't re-hit GitHub every time."""
    cache_path = cache_dir / f"{project_key}.csv"
    if cache_path.exists():
        return pd.read_csv(cache_path, encoding="utf-8", on_bad_lines="skip")

    url = f"{GITBUGS_RAW_BASE}/{config['path']}"
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()

    if config["path"].endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
            with zf.open(config["zip_inner"]) as f:
                df = pd.read_csv(f, encoding="utf-8", on_bad_lines="skip")
    else:
        df = pd.read_csv(io.StringIO(resp.content.decode("utf-8", errors="replace")), on_bad_lines="skip")

    df["project"] = project_key
    df.to_csv(cache_path, index=False)
    return df

GITBUGS_CACHE_DIR = DATA_DIR / "gitbugs_cache"
GITBUGS_CACHE_DIR.mkdir(exist_ok=True)

gitbugs_frames = []
for key, cfg in GITBUGS_PROJECTS.items():
    print(f"Fetching GitBugs project: {key}")
    df = fetch_gitbugs_project(key, cfg, GITBUGS_CACHE_DIR)
    gitbugs_frames.append(df)
    print(f"  {key}: {len(df):,} rows")

gitbugs_raw_df = pd.concat(gitbugs_frames, ignore_index=True)
print(f"Total GitBugs rows loaded: {len(gitbugs_raw_df):,}")

### Unify schema: bug-report-summarization pairs

Framed the same way as the code-summarization pairs: a long input
(`Description`) mapped to a short target (`Summary`). This mirrors the
code-to-docstring shape rather than inventing a `resolution_summary` field
that doesn't exist in the source data.

In [ ]:
def unify_gitbugs_row(row):
    return {
        "input_text": str(row.get("Description", "")).strip(),
        "target_text": str(row.get("Summary", "")).strip(),
        "source": "gitbugs",
        "language": "n/a",  # GitBugs is issue text, not tied to a programming language
        "task_type": "bug_report_summarization",
    }

def is_usable_gitbugs_row(row):
    desc = str(row.get("Description", ""))
    summ = str(row.get("Summary", ""))
    return desc.strip() not in ("", "nan") and summ.strip() not in ("", "nan")

usable_mask = gitbugs_raw_df.apply(is_usable_gitbugs_row, axis=1)
print(f"Usable GitBugs rows (non-empty Description and Summary): {usable_mask.sum():,} / {len(gitbugs_raw_df):,}")

gitbugs_unified = gitbugs_raw_df[usable_mask].apply(unify_gitbugs_row, axis=1, result_type="expand")
gitbugs_dataset = Dataset.from_pandas(gitbugs_unified.reset_index(drop=True))
print(f"Unified GitBugs dataset: {len(gitbugs_dataset):,} examples")

### Combine, dedupe, and filter to the ~200K target

Filtering criteria are logged explicitly here (not just applied silently) so
this step is reproducible and directly comparable to the ~200K figure used
for the Spring baseline's combined dataset scope.

In [ ]:
def dedupe_by_input_hash(dataset):
    """Drop exact-duplicate input_text records, keeping the first occurrence."""
    seen_hashes = set()
    keep_indices = []
    for i, text in enumerate(dataset["input_text"]):
        h = hashlib.md5(text.encode("utf-8")).hexdigest()
        if h not in seen_hashes:
            seen_hashes.add(h)
            keep_indices.append(i)
    return dataset.select(keep_indices)

def filter_by_length(dataset, min_input_tokens=3, min_target_tokens=3, max_input_chars=4000):
    """Drop empty/degenerate pairs and excessively long inputs that would
    dominate batch padding."""
    def is_valid(example):
        input_ok = min_input_tokens <= len(example["input_text"].split()) and len(example["input_text"]) <= max_input_chars
        target_ok = len(example["target_text"].split()) >= min_target_tokens
        return input_ok and target_ok
    return dataset.filter(is_valid)

TARGET_TOTAL = 200_000

combined = concatenate_datasets([code_dataset, gitbugs_dataset])

print(f"Before dedupe/filter: {len(combined):,}")
combined = dedupe_by_input_hash(combined)
print(f"After dedupe: {len(combined):,}")
combined = filter_by_length(combined)
print(f"After length filtering: {len(combined):,}")

# Stratified downsample to the ~200K target, proportional per (source, language)
# so no single language or source dominates the final combined set.
df = combined.to_pandas()
group_sizes = df.groupby(["source", "language"]).size()
group_fractions = group_sizes / group_sizes.sum()
target_per_group = (group_fractions * TARGET_TOTAL).round().astype(int)

sampled_parts = []
for (source, language), n_target in target_per_group.items():
    group_df = df[(df["source"] == source) & (df["language"] == language)]
    n_take = min(len(group_df), n_target)
    sampled_parts.append(group_df.sample(n=n_take, random_state=42))

final_df = pd.concat(sampled_parts).sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Final combined dataset: {len(final_df):,} examples (target was {TARGET_TOTAL:,})")
print(final_df.groupby(["source", "language"]).size())


### Save the combined dataset

In [ ]:
OUTPUT_PATH = DATA_DIR / "combined_dataset_v1.parquet"
final_df.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved combined dataset to {OUTPUT_PATH}")

# Log the exact filtering config alongside the data so this run is reproducible.
filter_log = {
    "target_total": TARGET_TOTAL,
    "final_total": len(final_df),
    "languages": LANGUAGES,
    "sources": sorted(final_df["source"].unique().tolist()),
    "min_input_tokens": 3,
    "min_target_tokens": 3,
    "max_input_chars": 4000,
    "dedupe_method": "md5 hash of input_text, first occurrence kept",
    "sampling": "stratified by (source, language), proportional to group size",
}
with open(DATA_DIR / "dataset_filter_log_v1.json", "w") as f:
    json.dump(filter_log, f, indent=2)
print("Saved filtering log.")


## Week 3-4: Infrastructure and Fairness Controls

### Carry-over compatibility fixes

Both fixes below are required for any new CodeT5 notebook, per the Spring
setup: the tokenizer loader works around CodeT5's RoBERTa-based tokenizer
being incompatible with newer Transformers versions, and `eval_strategy`
replaces the deprecated `evaluation_strategy` argument.

In [ ]:

def load_tokenizer_safe(model_name):
    """Carry-over fix from Spring, updated: CodeT5+ 220M's added_tokens.json
    trips a stricter type check in the fast tokenizer path on newer
    transformers/tokenizers versions (TypeError: Input must be a
    List[Union[str, AddedToken]]). Forcing the slow tokenizer avoids it;
    falling back to the explicit RobertaTokenizer class covers cases where
    AutoTokenizer still misbehaves.
    """
    try:
        return AutoTokenizer.from_pretrained(model_name, use_fast=False)
    except Exception as e:
        print(f"AutoTokenizer (slow) failed ({e}); falling back to RobertaTokenizer")
        from transformers import RobertaTokenizer
        return RobertaTokenizer.from_pretrained(model_name)


### Load CodeT5+ 220M base model

In [ ]:
# Imports and model name needed by this cell (previously defined only in a later cell)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq

MODEL_NAME = "Salesforce/codet5p-220m"


In [ ]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(f"Loaded {MODEL_NAME}: {base_model.num_parameters():,} parameters")


### Locked train/val/test split

This split is generated once here and saved to disk, then reused as-is for
the full fine-tune (Weeks 5-6), the LoRA run (Weeks 7-8), and, if the
decision is made to re-run it, CodeT5-small. Reusing the same split (not
regenerating per model) is what makes the eventual comparison fair.

In [ ]:
from sklearn.model_selection import train_test_split

SPLIT_SEED = 42
final_df = pd.read_parquet(DATA_DIR / "combined_dataset_v1.parquet")

train_df, temp_df = train_test_split(final_df, test_size=0.2, random_state=SPLIT_SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SPLIT_SEED)

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

SPLIT_DIR = DATA_DIR / "splits_v1"
SPLIT_DIR.mkdir(exist_ok=True)
train_df.to_parquet(SPLIT_DIR / "train.parquet", index=False)
val_df.to_parquet(SPLIT_DIR / "val.parquet", index=False)
test_df.to_parquet(SPLIT_DIR / "test.parquet", index=False)
print(f"Saved locked split to {SPLIT_DIR} (seed={SPLIT_SEED})")


### Tokenization / preprocessing function (shared across all model runs)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
import os
import json
from huggingface_hub import snapshot_download

def load_tokenizer_safe(model_name, cache_root="/content/model_cache"):
    local_dir = os.path.join(cache_root, model_name.replace("/", "__"))
    snapshot_download(repo_id=model_name, local_dir=local_dir)

    added_tokens_path = os.path.join(local_dir, "added_tokens.json")
    if os.path.exists(added_tokens_path):
        os.remove(added_tokens_path)

    special_tokens_path = os.path.join(local_dir, "special_tokens_map.json")
    if os.path.exists(special_tokens_path):
        with open(special_tokens_path, "r", encoding="utf-8") as f:
            special_tokens = json.load(f)
        additional = special_tokens.get("additional_special_tokens")
        if additional and isinstance(additional[0], dict):
            special_tokens["additional_special_tokens"] = [
                t["content"] if isinstance(t, dict) else t for t in additional
            ]
            with open(special_tokens_path, "w", encoding="utf-8") as f:
                json.dump(special_tokens, f)

    return AutoTokenizer.from_pretrained(local_dir)

MODEL_NAME = "Salesforce/codet5p-220m"
tokenizer = load_tokenizer_safe(MODEL_NAME)
print(f"Loaded tokenizer for {MODEL_NAME}")

In [ ]:
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

def preprocess_batch(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_ds = Dataset.from_pandas(train_df).map(preprocess_batch, batched=True, remove_columns=train_df.columns.tolist())
val_ds = Dataset.from_pandas(val_df).map(preprocess_batch, batched=True, remove_columns=val_df.columns.tolist())
test_ds = Dataset.from_pandas(test_df).map(preprocess_batch, batched=True, remove_columns=test_df.columns.tolist())

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=base_model)
print("Tokenized train/val/test sets ready.")

In [ ]:
print(tokenizer, base_model.num_parameters(), len(train_df))

### Shared eval harness (BLEU-4, ROUGE-L, METEOR)

Built once here and reused unchanged for CodeT5+ full fine-tune, CodeT5+ with
LoRA, and CodeT5-small if it's re-run, so metric computation isn't a source
of inconsistency between the three models.

Also worth re-checking once real predictions come in: whether the near-ceiling
scores seen in Spring (an artifact of the deterministic logic suffix from
`extract_logic_summary()`) still show up in this multi-language setting.

In [ ]:
import evaluate
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("punkt", quiet=True)

bleu_metric = evaluate.load("sacrebleu")
rouge_metric = evaluate.load("rouge")
meteor_metric = evaluate.load("meteor")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = [[(l if l != -100 else tokenizer.pad_token_id) for l in label] for label in labels]
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    bleu = bleu_metric.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
    rouge = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
    meteor = meteor_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "bleu4": bleu["score"],
        "rougeL": rouge["rougeL"] * 100,
        "meteor": meteor["meteor"] * 100,
    }

print("Eval harness ready: BLEU-4, ROUGE-L, METEOR.")


### Shared TrainingArguments template

Uses `eval_strategy` (not the deprecated `evaluation_strategy`). This same
template, with only `output_dir`, `run_name`, and LoRA-specific flags varying,
is what Weeks 5-6 and 7-8 each build on, so epoch count and compute budget
stay matched between the full fine-tune and the LoRA run.

In [ ]:
def build_training_args(output_dir, run_name, num_train_epochs=3, learning_rate=5e-5):
    return TrainingArguments(
        output_dir=output_dir,
        run_name=run_name,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        num_train_epochs=num_train_epochs,
        weight_decay=0.01,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        fp16=True,
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="bleu4",
        report_to="none",
        seed=SPLIT_SEED,
    )

print("build_training_args() ready. Weeks 5-6 and 7-8 both call this with matched epochs/lr.")


### LoRA config, staged for Weeks 7-8

Set up here so the config is version-controlled alongside the rest of the
pipeline; the actual LoRA training run happens in Weeks 7-8, not this
notebook.

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q", "v"],  # CodeT5+ attention projections
    bias="none",
)

# Applied when the Weeks 7-8 LoRA run starts, not here:
# lora_model = get_peft_model(base_model, lora_config)
# lora_model.print_trainable_parameters()

print("LoRA config staged for Weeks 7-8.")
print(lora_config)


## Summary / handoff to Weeks 5-8

- Combined multi-language dataset saved to `combined_dataset_v1.parquet`, with the filtering log alongside it for reproducibility.
- Locked train/val/test split saved to `splits_v1/`, seed 42. Weeks 5-6, 7-8, and any CodeT5-small re-run all load this split rather than regenerating it.
- `preprocess_batch`, `compute_metrics`, and `build_training_args` are the three shared functions Weeks 5-8 build directly on top of.
- GitBugs now loads live from the real `av9ash/gitbugs` repo (CC BY 4.0), Mozilla Core excluded since only duplicate-id pairs are available for it, not bug text.
- **Still open before Week 5 starts:** confirm with Dr. Adigun whether CodeT5-small gets re-trained on this full split or stays as the Spring 50K baseline.


### Verify outputs are on Drive


In [ ]:
import os
for root, dirs, files in os.walk('/content/drive/MyDrive/capstone/data'):
    if 'gitbugs' in root.lower() and root.endswith('cache'):
        continue
    for f in files:
        p = os.path.join(root, f)
        print(f'{os.path.getsize(p)/1e6:8.1f} MB  {p}')
